# 🏠 Satellite Property Valuation - Training Notebook

This notebook trains the multimodal fusion model combining satellite imagery with tabular property data.

**Requirements:**
- Google Colab with GPU runtime
- Uploaded data files (train.csv, satellite images)

---

## 1️⃣ Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install required packages
!pip install torch torchvision timm pytorch-grad-cam pandas scikit-learn tqdm matplotlib seaborn pillow -q

In [ ]:
# Imports
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.transforms as transforms
from torchvision.models import resnet18, ResNet18_Weights

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2️⃣ Upload Data

Upload the following files:
- `train.csv` (processed training data)
- `images.zip` (satellite images)

In [ ]:
# Upload files (uncomment to use)
# from google.colab import files
# uploaded = files.upload()

In [ ]:
# Or mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Set paths (modify as needed)
DATA_DIR = Path('/content/drive/MyDrive/satellite-property-valuation/data')
TRAIN_CSV = DATA_DIR / 'processed' / 'train.csv'
IMAGE_DIR = DATA_DIR / 'images' / 'train'
MODELS_DIR = Path('/content/drive/MyDrive/satellite-property-valuation/models')
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## 3️⃣ Model Architecture

In [ ]:
class ImageEncoder(nn.Module):
    """CNN encoder using pretrained ResNet-18."""
    
    def __init__(self, embedding_dim=512, pretrained=True):
        super().__init__()
        
        if pretrained:
            self.backbone = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
        else:
            self.backbone = resnet18(weights=None)
        
        backbone_out = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()
        
        if embedding_dim != backbone_out:
            self.projection = nn.Sequential(
                nn.Linear(backbone_out, embedding_dim),
                nn.ReLU(),
                nn.Dropout(0.2)
            )
        else:
            self.projection = nn.Identity()
    
    def forward(self, x):
        features = self.backbone(x)
        return self.projection(features)


class TabularEncoder(nn.Module):
    """MLP encoder for tabular features."""
    
    def __init__(self, input_dim, hidden_dims=[64, 32], embedding_dim=32, dropout=0.2):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for hidden_dim in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = hidden_dim
        
        layers.append(nn.Linear(prev_dim, embedding_dim))
        self.encoder = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.encoder(x)


class FusionModel(nn.Module):
    """Multimodal fusion model."""
    
    def __init__(self, tabular_input_dim, image_embedding_dim=512, 
                 tabular_embedding_dim=32, fusion_hidden_dims=[256, 128],
                 dropout=0.3, pretrained=True):
        super().__init__()
        
        self.image_encoder = ImageEncoder(image_embedding_dim, pretrained)
        self.tabular_encoder = TabularEncoder(tabular_input_dim, embedding_dim=tabular_embedding_dim)
        
        fusion_input_dim = image_embedding_dim + tabular_embedding_dim
        fusion_layers = []
        prev_dim = fusion_input_dim
        
        for hidden_dim in fusion_hidden_dims:
            fusion_layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(dropout)
            ])
            prev_dim = hidden_dim
        
        fusion_layers.append(nn.Linear(prev_dim, 1))
        self.fusion = nn.Sequential(*fusion_layers)
    
    def forward(self, image, tabular):
        img_emb = self.image_encoder(image)
        tab_emb = self.tabular_encoder(tabular)
        combined = torch.cat([img_emb, tab_emb], dim=1)
        return self.fusion(combined)

## 4️⃣ Dataset Class

In [ ]:
class PropertyDataset(Dataset):
    """Dataset for property valuation."""
    
    def __init__(self, csv_path, image_dir, feature_columns=None, 
                 scaler=None, transform=None, is_train=True):
        self.df = pd.read_csv(csv_path)
        self.image_dir = Path(image_dir)
        self.is_train = is_train
        
        # Default transform
        if transform is None:
            self.transform = transforms.Compose([
                transforms.Resize((224, 224)),
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            ])
        else:
            self.transform = transform
        
        # Feature columns
        if feature_columns is None:
            exclude = ['id', 'price', 'date', 'lat', 'long']
            self.feature_columns = [c for c in self.df.columns 
                                    if c not in exclude 
                                    and self.df[c].dtype in [np.int64, np.float64]]
        else:
            self.feature_columns = feature_columns
        
        # Tabular features
        self.tabular = self.df[self.feature_columns].values.astype(np.float32)
        
        # Scaler
        if scaler is None and is_train:
            self.scaler = StandardScaler()
            self.tabular = self.scaler.fit_transform(self.tabular)
        elif scaler is not None:
            self.scaler = scaler
            self.tabular = self.scaler.transform(self.tabular)
        else:
            self.scaler = None
        
        # Prices
        self.prices = self.df['price'].values.astype(np.float32) if 'price' in self.df.columns else None
        self.ids = self.df['id'].values
        
        # Placeholder for missing images
        self.placeholder = torch.zeros(3, 224, 224)
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        # Load image
        img_path = self.image_dir / f"{self.ids[idx]}.png"
        if img_path.exists():
            try:
                img = Image.open(img_path).convert('RGB')
                img_tensor = self.transform(img)
            except:
                img_tensor = self.placeholder
        else:
            img_tensor = self.placeholder
        
        # Tabular
        tab_tensor = torch.from_numpy(self.tabular[idx])
        
        # Price
        price = torch.tensor(self.prices[idx]) if self.prices is not None else torch.tensor(0.0)
        
        return img_tensor, tab_tensor, price
    
    def get_feature_dim(self):
        return len(self.feature_columns)

## 5️⃣ Training Functions

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    
    for images, tabular, prices in tqdm(train_loader, desc='Training'):
        images = images.to(device)
        tabular = tabular.to(device)
        prices = prices.to(device).unsqueeze(1)
        
        optimizer.zero_grad()
        outputs = model(images, tabular)
        loss = criterion(outputs, prices)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item() * len(prices)
    
    return total_loss / len(train_loader.dataset)


@torch.no_grad()
def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds, all_targets = [], []
    
    for images, tabular, prices in val_loader:
        images = images.to(device)
        tabular = tabular.to(device)
        prices = prices.to(device).unsqueeze(1)
        
        outputs = model(images, tabular)
        loss = criterion(outputs, prices)
        
        total_loss += loss.item() * len(prices)
        all_preds.extend(outputs.cpu().numpy().flatten())
        all_targets.extend(prices.cpu().numpy().flatten())
    
    avg_loss = total_loss / len(val_loader.dataset)
    rmse = np.sqrt(avg_loss)
    
    # R²
    all_preds, all_targets = np.array(all_preds), np.array(all_targets)
    ss_res = np.sum((all_targets - all_preds) ** 2)
    ss_tot = np.sum((all_targets - all_targets.mean()) ** 2)
    r2 = 1 - (ss_res / ss_tot)
    
    return avg_loss, rmse, r2

## 6️⃣ Load Data & Create Model

In [ ]:
# Load dataset
full_dataset = PropertyDataset(
    csv_path=TRAIN_CSV,
    image_dir=IMAGE_DIR,
    is_train=True
)

print(f'Dataset size: {len(full_dataset)}')
print(f'Feature dimension: {full_dataset.get_feature_dim()}')
print(f'Features: {full_dataset.feature_columns}')

In [ ]:
# Split dataset
val_size = int(0.2 * len(full_dataset))
train_size = len(full_dataset) - val_size

train_dataset, val_dataset = random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

print(f'Training: {len(train_dataset)}, Validation: {len(val_dataset)}')

In [ ]:
# Create dataloaders
BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, 
                          num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=2, pin_memory=True)

In [ ]:
# Create model
model = FusionModel(
    tabular_input_dim=full_dataset.get_feature_dim(),
    pretrained=True
).to(device)

print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

## 7️⃣ Train Model

In [ ]:
# Training config
NUM_EPOCHS = 50
LEARNING_RATE = 1e-4

criterion = nn.MSELoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', 
                                                   factor=0.5, patience=3, verbose=True)

# Training history
history = {'train_loss': [], 'val_loss': [], 'val_rmse': [], 'val_r2': []}
best_val_loss = float('inf')

In [ ]:
# Training loop
for epoch in range(1, NUM_EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_rmse, val_r2 = validate(model, val_loader, criterion, device)
    
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_rmse'].append(val_rmse)
    history['val_r2'].append(val_r2)
    
    scheduler.step(val_loss)
    
    print(f'Epoch {epoch}/{NUM_EPOCHS}')
    print(f'  Train Loss: {train_loss:.4f}, Train RMSE: ${np.sqrt(train_loss):,.0f}')
    print(f'  Val Loss: {val_loss:.4f}, Val RMSE: ${val_rmse:,.0f}, Val R²: {val_r2:.4f}')
    
    # Save best model
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'val_rmse': val_rmse,
            'val_r2': val_r2,
            'history': history
        }, MODELS_DIR / 'fusion_model.pth')
        print(f'  ✓ Saved best model!')
    
    print()

## 8️⃣ Plot Training Results

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss
axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Validation')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()

# RMSE
axes[1].plot(history['val_rmse'])
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('RMSE ($)')
axes[1].set_title('Validation RMSE')

# R²
axes[2].plot(history['val_r2'])
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('R²')
axes[2].set_title('Validation R²')

plt.tight_layout()
plt.savefig(MODELS_DIR / 'training_curves.png', dpi=150)
plt.show()

## 9️⃣ Final Results

In [ ]:
print('='*50)
print('TRAINING COMPLETE')
print('='*50)
print(f"Best Validation RMSE: ${min(history['val_rmse']):,.0f}")
print(f"Best Validation R²: {max(history['val_r2']):.4f}")
print(f"\nModel saved to: {MODELS_DIR / 'fusion_model.pth'}")